# Paper 4: NRR-IME Complete Experiments v5

**Key changes from v4:**
- Phase 1.0 now runs ALL models × ALL scenarios (was Claude-only for Spring/Court)
- All inputs are English across all scenarios for cross-model comparability
- 45 experiments × 3 trials = 135 API session runs

**Carried over from v4:**
- Temperature: 0.3 (realistic operating conditions)
- 3 trials per experiment → mean ± std for all metrics
- Post-hoc analyses: weight volatility (Phase 1.3), operator distribution (Phase 1.5)

## Design Space

| Phase | State sent | LLM returns | Failure mode |
|-------|-----------|-------------|--------------|
| 1.0 | Full JSON | Full JSON | High variance, unstable |
| 1.3 | Labels only | Numeric weights | Bang-bang oscillation |
| 1.5 | Labels only | Binary operator | **None (optimal)** |
| 1.6 | Minimal | Binary operator | Universal extraction failure |
| 2.0 | Labels only | Single interpretation | Semantic collapse |
| 3.0 | Hybrid | Auto or operator | Input dependency |

## Experiment Matrix: 45 × 3 = 135 runs

| Phase | Scenarios | Models | Experiments | Runs |
|-------|-----------|--------|-------------|------|
| 1.0 | Bank, Spring, Court | 3 | 9 | 27 |
| 1.3 | Bank | 3 | 3 | 9 |
| 1.5 | Bank, Spring, Court | 3 | 9 | 27 |
| 1.6 | Bank | 3 | 3 | 9 |
| 2.0 | Bank | 3 | 3 | 9 |
| 3.0 Explicit | Bank, Spring, Court | 3 | 9 | 27 |
| 3.0 Ambiguous | Bank, Spring, Court | 3 | 9 | 27 |
| **Total** | | | **45** | **135** |

In [ ]:
!pip install anthropic openai google-generativeai numpy -q

In [ ]:
import anthropic
import openai
import google.generativeai as genai
import json
import time
import re
import numpy as np
from datetime import datetime
from typing import Dict, List, Tuple, Optional

try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')
except:
    import os
    ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')
    GEMINI_API_KEY = os.environ.get('GOOGLE_API_KEY', '')

anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)
genai.configure(api_key=GEMINI_API_KEY)
print("✓ API clients initialized")

In [ ]:
MODELS = {
    'claude': 'claude-sonnet-4-20250514',
    'gpt': 'gpt-4o-mini-2024-07-18',
    'gemini': 'gemini-2.0-flash'
}
MAX_TOKENS = 200
TEMPERATURE = 0.3
ALPHA = 0.4
N_TRIALS = 3
print(f"Models: {MODELS}")
print(f"Parameters: max_tokens={MAX_TOKENS}, temp={TEMPERATURE}, alpha={ALPHA}, trials={N_TRIALS}")

## Scenarios

In [ ]:
BANK_SCENARIO = {
    'name': 'bank', 'turns': 5,
    'interpretations': {'financial': 0.5, 'river': 0.5},
    'inputs': [
        "The bank is solid and reliable.",
        "Perfect for ducks and herons.",
        "Their interest rates just dropped.",
        "The muddy bank eroded last spring.",
        "I deposited my paycheck there."
    ]
}
BANK_AMBIGUOUS = {
    'name': 'bank_ambiguous', 'turns': 5,
    'interpretations': {'financial': 0.5, 'river': 0.5},
    'inputs': [
        "The bank is solid and reliable.",
        "It's a peaceful place with shade.",
        "They offer great rates.",
        "I walked along the edge yesterday.",
        "The place has been here for decades."
    ]
}
SPRING_SCENARIO = {
    'name': 'spring', 'turns': 10,
    'interpretations': {'season': 0.4, 'water': 0.3, 'mechanical': 0.2, 'leap': 0.1},
    'inputs': [
        "Flowers bloom in spring.",
        "The fountain water is cold and clear.",
        "The spring-loaded toy popped open.",
        "The cat sprang across the room.",
        "I love the cherry blossom season.",
        "We visited a natural geyser.",
        "She wore a light coat for the season.",
        "The mattress needs replacement.",
        "The spring equinox is approaching.",
        "The coil finally snapped."
    ]
}
SPRING_AMBIGUOUS = {
    'name': 'spring_ambiguous', 'turns': 5,
    'interpretations': {'season': 0.4, 'water': 0.3, 'mechanical': 0.2, 'leap': 0.1},
    'inputs': [
        "Time moves in cycles and renews.",
        "Something bubbled up from deep below.",
        "The material has great tension and resilience.",
        "She bounded across the stage gracefully.",
        "A period of fresh beginnings arrived."
    ]
}
COURT_SCENARIO = {
    'name': 'court', 'turns': 12,
    'interpretations': {'law': 0.35, 'sports': 0.30, 'royal': 0.25, 'romance': 0.10},
    'inputs': [
        "A tennis match was scheduled.",
        "The testimony was compelling.",
        "The palace ceremony was grand.",
        "The suitor brought gifts.",
        "The basketball court was freshly painted.",
        "The verdict shocked everyone.",
        "The throne room echoed with music.",
        "Traditional courtship rituals were observed.",
        "The judge made a ruling.",
        "She served an ace on the tennis court.",
        "The king held a grand ceremony.",
        "The lawyer presented the evidence."
    ]
}
COURT_AMBIGUOUS = {
    'name': 'court_ambiguous', 'turns': 5,
    'interpretations': {'law': 0.35, 'sports': 0.30, 'royal': 0.25, 'romance': 0.10},
    'inputs': [
        "I need to settle this dispute.",
        "The competition was fierce.",
        "A grand formal gathering took place.",
        "He tried to win her favor.",
        "Rules and protocols must be followed."
    ]
}
print("✓ All scenarios defined")

## Keyword Dictionaries + Validation

In [ ]:
BANK_KEYWORDS = {
    'financial': ['interest rate', 'deposit', 'paycheck', 'account', 'loan',
                  'money', 'withdraw', 'atm', 'credit'],
    'river': ['duck', 'heron', 'muddy', 'eroded', 'erosion', 'flood',
              'creek', 'shore', 'stream', 'fish']
}
SPRING_KEYWORDS = {
    'season': ['bloom', 'flower', 'equinox', 'cherry', 'blossom'],
    'water': ['fountain', 'geyser', 'well water', 'mineral water'],
    'mechanical': ['spring-loaded', 'coil', 'mattress', 'snap'],
    'leap': ['sprang', 'jumped', 'leapt', 'vault']
}
COURT_KEYWORDS = {
    'law': ['verdict', 'judge', 'legal', 'lawyer', 'testimony', 'trial'],
    'sports': ['tennis', 'basketball', 'match', 'racket', 'serve an ace'],
    'royal': ['palace', 'throne', 'ceremony', 'king', 'queen', 'crown'],
    'romance': ['courtship', 'woo', 'suitor', 'admirer']
}
KEYWORD_MAP = {
    'bank': BANK_KEYWORDS, 'bank_ambiguous': BANK_KEYWORDS,
    'spring': SPRING_KEYWORDS, 'spring_ambiguous': SPRING_KEYWORDS,
    'court': COURT_KEYWORDS, 'court_ambiguous': COURT_KEYWORDS
}

def keyword_match(text, keywords):
    text_lower = text.lower()
    for interp, kw_list in keywords.items():
        for kw in kw_list:
            if kw.lower() in text_lower:
                return interp, kw
    return None, None

all_valid = True
for name, scenario in [('bank_ambiguous', BANK_AMBIGUOUS),
                        ('spring_ambiguous', SPRING_AMBIGUOUS),
                        ('court_ambiguous', COURT_AMBIGUOUS)]:
    for inp in scenario['inputs']:
        interp, kw = keyword_match(inp, KEYWORD_MAP[name])
        if interp:
            print(f"  ✗ LEAK: '{inp}' matches '{kw}' → {interp}")
            all_valid = False
if all_valid:
    print("✓ No keyword leaks in ambiguous scenarios")
else:
    raise ValueError("Keyword leaks detected")

## API + Parser

In [ ]:
def call_claude(prompt, max_retries=3):
    for attempt in range(max_retries):
        try:
            r = anthropic_client.messages.create(
                model=MODELS['claude'], max_tokens=MAX_TOKENS, temperature=TEMPERATURE,
                messages=[{"role": "user", "content": prompt}])
            return r.content[0].text, r.usage.input_tokens, r.usage.output_tokens
        except Exception as e:
            if attempt < max_retries - 1: time.sleep(2 ** attempt); continue
            raise

def call_gpt(prompt, max_retries=3):
    for attempt in range(max_retries):
        try:
            r = openai_client.chat.completions.create(
                model=MODELS['gpt'], max_tokens=MAX_TOKENS, temperature=TEMPERATURE,
                messages=[{"role": "user", "content": prompt}])
            return r.choices[0].message.content, r.usage.prompt_tokens, r.usage.completion_tokens
        except Exception as e:
            if attempt < max_retries - 1: time.sleep(2 ** attempt); continue
            raise

def call_gemini(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            model = genai.GenerativeModel(MODELS['gemini'])
            config = genai.types.GenerationConfig(max_output_tokens=MAX_TOKENS, temperature=TEMPERATURE)
            r = model.generate_content(prompt, generation_config=config)
            return r.text, r.usage_metadata.prompt_token_count, r.usage_metadata.candidates_token_count
        except Exception as e:
            if attempt < max_retries - 1: time.sleep(2 ** attempt + 1); continue
            raise

def call_model(model_name, prompt):
    if model_name == 'claude': return call_claude(prompt)
    elif model_name == 'gpt': return call_gpt(prompt)
    elif model_name == 'gemini': return call_gemini(prompt)
    raise ValueError(f"Unknown model: {model_name}")

def extract_operator(resp, valid_targets):
    operator, target = None, None
    resp_clean = resp.lower().replace('*', '').replace('`', '').strip()
    for line in resp_clean.split('\n'):
        line = line.strip()
        if line.startswith('operator:'):
            val = line.split(':', 1)[1].strip()
            if 'sigma' in val: operator = 'sigma'
            elif 'delta' in val: operator = 'delta'
            elif not operator:
                for t in valid_targets:
                    if t.lower() in val: target = t; break
        if line.startswith('target:'):
            val = line.split(':', 1)[1].strip()
            for t in valid_targets:
                if t.lower() in val: target = t; break
            if 'sigma' in val: operator = 'sigma'
            elif 'delta' in val: operator = 'delta'
    if not operator:
        if 'sigma' in resp_clean: operator = 'sigma'
        elif 'delta' in resp_clean: operator = 'delta'
    if not target:
        for t in valid_targets:
            if t.lower() in resp_clean: target = t; break
    return operator, target

print("✓ API functions + parser defined")

## Phase Functions

In [ ]:
def run_phase_10(scenario, model_name):
    state = {"interpretations": scenario['interpretations'].copy()}
    results = {'phase': '1.0', 'scenario': scenario['name'], 'model': model_name,
               'turns': [], 'total_tokens': 0, 'total_input': 0, 'total_output': 0}
    for i, text in enumerate(scenario['inputs'], 1):
        prompt = f"""Current NRR state (JSON):
{json.dumps(state, indent=2)}

New input: "{text}"
Update interpretation weights. Return complete updated state as JSON."""
        resp, inp, out = call_model(model_name, prompt)
        try: state = json.loads(resp)
        except: pass
        results['turns'].append({'turn': i, 'input_tokens': inp, 'output_tokens': out, 'total_tokens': inp+out, 'response': resp[:200]})
        results['total_tokens'] += inp+out; results['total_input'] += inp; results['total_output'] += out
        time.sleep(0.3)
    results['avg_per_turn'] = results['total_tokens'] / len(scenario['inputs'])
    return results

def run_phase_13(scenario, model_name):
    state = scenario['interpretations'].copy()
    valid_keys = list(state.keys())
    results = {'phase': '1.3', 'scenario': scenario['name'], 'model': model_name,
               'turns': [], 'total_tokens': 0, 'total_input': 0, 'total_output': 0,
               'weight_parse_success': 0, 'weight_histories': []}
    for i, text in enumerate(scenario['inputs'], 1):
        weights_str = ", ".join(valid_keys)
        prompt = f"""Interpretations: [{weights_str}]

User input: "{text}"

Assign updated probability weights to each interpretation.
Weights must sum to 1.0.

Output ONLY in this format (one per line):
{chr(10).join(f'{k}: <weight>' for k in valid_keys)}"""
        resp, inp, out = call_model(model_name, prompt)
        parsed = {}
        for line in resp.split('\n'):
            line = line.strip().lower().replace('*', '')
            for k in valid_keys:
                if k.lower() in line:
                    nums = re.findall(r'[0-9]+\.?[0-9]*', line)
                    if nums: parsed[k] = float(nums[0]); break
        parse_ok = len(parsed) == len(valid_keys) and abs(sum(parsed.values()) - 1.0) < 0.05
        if parse_ok:
            results['weight_parse_success'] += 1
            state = parsed
        results['weight_histories'].append(dict(state))
        results['turns'].append({'turn': i, 'input_tokens': inp, 'output_tokens': out,
            'total_tokens': inp+out, 'parsed_weights': parsed, 'parse_ok': parse_ok, 'response': resp[:200]})
        results['total_tokens'] += inp+out; results['total_input'] += inp; results['total_output'] += out
        time.sleep(0.3)
    n = len(scenario['inputs'])
    results['avg_per_turn'] = results['total_tokens'] / n
    results['weight_parse_rate'] = results['weight_parse_success'] / n
    # Weight volatility: mean absolute change between consecutive turns
    if len(results['weight_histories']) >= 2:
        deltas = []
        for j in range(1, len(results['weight_histories'])):
            prev = results['weight_histories'][j-1]
            curr = results['weight_histories'][j]
            for k in valid_keys:
                if k in prev and k in curr:
                    deltas.append(abs(curr[k] - prev[k]))
        results['weight_volatility'] = float(np.mean(deltas)) if deltas else 0.0
    return results

def run_phase_15(scenario, model_name):
    state = scenario['interpretations'].copy()
    state_id = f"state_{scenario['name']}_001"
    results = {'phase': '1.5', 'scenario': scenario['name'], 'model': model_name,
               'turns': [], 'total_tokens': 0, 'total_input': 0, 'total_output': 0,
               'extraction_success': 0, 'sigma_count': 0, 'delta_count': 0}
    for i, text in enumerate(scenario['inputs'], 1):
        prompt = f"""Current state: {state_id}
Interpretations: {list(state.keys())}

User input: "{text}"

Available operators:
- sigma (strengthen): Increase weight of target interpretation
- delta (dampen): Decrease weight of target interpretation

Output ONLY in this format:
operator: <sigma or delta>
target: <interpretation_key>"""
        resp, inp, out = call_model(model_name, prompt)
        op, tgt = extract_operator(resp, list(state.keys()))
        ok = op is not None and tgt is not None and tgt in state
        if ok:
            results['extraction_success'] += 1
            if op == 'sigma': results['sigma_count'] += 1; state[tgt] = min(1.0, state[tgt] + ALPHA)
            elif op == 'delta': results['delta_count'] += 1; state[tgt] = max(0.0, state[tgt] - ALPHA)
            total_w = sum(state.values())
            if total_w > 0: state = {k: v/total_w for k, v in state.items()}
        results['turns'].append({'turn': i, 'input_tokens': inp, 'output_tokens': out,
            'total_tokens': inp+out, 'operator': op, 'target': tgt, 'success': ok, 'response': resp[:200]})
        results['total_tokens'] += inp+out; results['total_input'] += inp; results['total_output'] += out
        time.sleep(0.3)
    n = len(scenario['inputs'])
    results['avg_per_turn'] = results['total_tokens'] / n
    results['extraction_rate'] = results['extraction_success'] / n
    return results

def run_phase_16(scenario, model_name):
    state = scenario['interpretations'].copy()
    results = {'phase': '1.6', 'scenario': scenario['name'], 'model': model_name,
               'turns': [], 'total_tokens': 0, 'total_input': 0, 'total_output': 0,
               'extraction_success': 0}
    for i, text in enumerate(scenario['inputs'], 1):
        prompt = f"""{list(state.keys())}
"{text}"
op? tgt?"""
        resp, inp, out = call_model(model_name, prompt)
        op, tgt = extract_operator(resp, list(state.keys()))
        ok = op is not None and tgt is not None
        if ok: results['extraction_success'] += 1
        results['turns'].append({'turn': i, 'input_tokens': inp, 'output_tokens': out,
            'total_tokens': inp+out, 'operator': op, 'target': tgt, 'success': ok, 'response': resp[:200]})
        results['total_tokens'] += inp+out; results['total_input'] += inp; results['total_output'] += out
        time.sleep(0.3)
    n = len(scenario['inputs'])
    results['avg_per_turn'] = results['total_tokens'] / n
    results['extraction_rate'] = results['extraction_success'] / n
    return results

def run_phase_20(scenario, model_name):
    interps = list(scenario['interpretations'].keys())
    results = {'phase': '2.0', 'scenario': scenario['name'], 'model': model_name,
               'turns': [], 'total_tokens': 0, 'total_input': 0, 'total_output': 0,
               'single_interpretation_count': 0}
    p1 = f"""Generate a semantic embedding description for these interpretations:
{json.dumps(interps)}
Provide a brief semantic description for each."""
    resp, inp, out = call_model(model_name, p1)
    results['turns'].append({'turn': 1, 'input_tokens': inp, 'output_tokens': out,
        'total_tokens': inp+out, 'response': resp[:200]})
    results['total_tokens'] += inp+out; results['total_input'] += inp; results['total_output'] += out
    for i, text in enumerate(scenario['inputs'][1:], 2):
        p2 = f"""Which interpretation is most similar?
Input: "{text}"
Interpretations: {interps}
Output ONLY the interpretation name."""
        r2, i2, o2 = call_model(model_name, p2)
        r2_lower = r2.strip().lower()
        single = any(r2_lower.startswith(x.lower()) or r2_lower == x.lower() for x in interps)
        if single: results['single_interpretation_count'] += 1
        results['turns'].append({'turn': i, 'input_tokens': i2, 'output_tokens': o2,
            'total_tokens': i2+o2, 'response': r2[:200], 'returned_single': single})
        results['total_tokens'] += i2+o2; results['total_input'] += i2; results['total_output'] += o2
        time.sleep(0.3)
    n = len(scenario['inputs'])
    results['avg_per_turn'] = results['total_tokens'] / n
    results['single_rate'] = results['single_interpretation_count'] / max(1, n-1)
    return results

def run_phase_30(scenario, keywords, model_name, is_ambiguous=False):
    mode = 'ambiguous' if is_ambiguous else 'explicit'
    state = scenario['interpretations'].copy()
    state_id = f"state_{scenario['name']}_001"
    results = {'phase': '3.0', 'scenario': scenario['name'], 'mode': mode,
               'model': model_name, 'turns': [], 'total_tokens': 0,
               'total_input': 0, 'total_output': 0, 'llm_calls': 0, 'auto_calls': 0}
    for i, text in enumerate(scenario['inputs'], 1):
        mi, mk = keyword_match(text, keywords)
        if mi:
            results['turns'].append({'turn': i, 'total_tokens': 0, 'method': 'AUTO',
                'matched_interpretation': mi, 'matched_keyword': mk})
            results['auto_calls'] += 1
            state[mi] = min(1.0, state[mi] + ALPHA)
            total_w = sum(state.values())
            if total_w > 0: state = {k: v/total_w for k, v in state.items()}
        else:
            prompt = f"""Current state: {state_id}
Interpretations: {list(state.keys())}

User input: "{text}"

Available operators:
- sigma (strengthen): Increase weight of target interpretation
- delta (dampen): Decrease weight of target interpretation

Output ONLY in this format:
operator: <sigma or delta>
target: <interpretation_key>"""
            resp, inp, out = call_model(model_name, prompt)
            results['turns'].append({'turn': i, 'input_tokens': inp, 'output_tokens': out,
                'total_tokens': inp+out, 'method': 'LLM', 'response': resp[:200]})
            results['total_tokens'] += inp+out; results['total_input'] += inp; results['total_output'] += out
            results['llm_calls'] += 1
            time.sleep(0.3)
    n = len(scenario['inputs'])
    results['avg_per_turn'] = results['total_tokens'] / n
    results['auto_rate'] = results['auto_calls'] / n
    return results

print("✓ All phase functions defined (1.0, 1.3, 1.5, 1.6, 2.0, 3.0)")

## Run All (45 experiments × 3 trials)

In [ ]:
def run_all():
    models = ['claude', 'gpt', 'gemini']
    all_results = {
        'metadata': {
            'date': datetime.now().isoformat(),
            'version': 'v5',
            'models': MODELS,
            'parameters': {'max_tokens': MAX_TOKENS, 'temperature': TEMPERATURE, 'alpha': ALPHA, 'n_trials': N_TRIALS}
        },
        'experiments': []
    }
    run_count = 0
    total_runs = 45 * N_TRIALS

    def run_with_trials(fn, *args, label="", **kwargs):
        nonlocal run_count
        trials = []
        for t in range(1, N_TRIALS + 1):
            run_count += 1
            try:
                r = fn(*args, **kwargs)
                r['trial'] = t
                trials.append(r)
                all_results['experiments'].append(r)
                print(f"    [{run_count}/{total_runs}] trial {t}: {r['total_tokens']} tok")
            except Exception as e:
                print(f"    [{run_count}/{total_runs}] trial {t}: ✗ ERROR: {e}")
            time.sleep(0.5)
        return trials

    # [1] Phase 1.0 — ALL models × ALL scenarios
    print("=" * 60)
    print("[1/7] Phase 1.0 Baseline (3 models × 3 scenarios)")
    print("=" * 60)
    for s in [BANK_SCENARIO, SPRING_SCENARIO, COURT_SCENARIO]:
        for m in models:
            print(f"  Phase 1.0 - {m} - {s['name']}:")
            run_with_trials(run_phase_10, s, m)

    # [2] Phase 1.3
    print("\n" + "=" * 60)
    print("[2/7] Phase 1.3 Numeric Weights")
    print("=" * 60)
    for m in models:
        print(f"  Phase 1.3 - {m} - bank:")
        run_with_trials(run_phase_13, BANK_SCENARIO, m)

    # [3] Phase 1.5
    print("\n" + "=" * 60)
    print("[3/7] Phase 1.5 Operator")
    print("=" * 60)
    for s in [BANK_SCENARIO, SPRING_SCENARIO, COURT_SCENARIO]:
        for m in models:
            print(f"  Phase 1.5 - {m} - {s['name']}:")
            run_with_trials(run_phase_15, s, m)

    # [4] Phase 1.6
    print("\n" + "=" * 60)
    print("[4/7] Phase 1.6 Over-compressed")
    print("=" * 60)
    for m in models:
        print(f"  Phase 1.6 - {m} - bank:")
        run_with_trials(run_phase_16, BANK_SCENARIO, m)

    # [5] Phase 2.0
    print("\n" + "=" * 60)
    print("[5/7] Phase 2.0 Embedding")
    print("=" * 60)
    for m in models:
        print(f"  Phase 2.0 - {m} - bank:")
        run_with_trials(run_phase_20, BANK_SCENARIO, m)

    # [6] Phase 3.0 Explicit
    print("\n" + "=" * 60)
    print("[6/7] Phase 3.0 Explicit")
    print("=" * 60)
    for s, kw in [(BANK_SCENARIO, BANK_KEYWORDS), (SPRING_SCENARIO, SPRING_KEYWORDS), (COURT_SCENARIO, COURT_KEYWORDS)]:
        for m in models:
            print(f"  Phase 3.0 exp - {m} - {s['name']}:")
            run_with_trials(run_phase_30, s, kw, m, False)

    # [7] Phase 3.0 Ambiguous
    print("\n" + "=" * 60)
    print("[7/7] Phase 3.0 Ambiguous")
    print("=" * 60)
    for s, kw in [(BANK_AMBIGUOUS, BANK_KEYWORDS), (SPRING_AMBIGUOUS, SPRING_KEYWORDS), (COURT_AMBIGUOUS, COURT_KEYWORDS)]:
        for m in models:
            print(f"  Phase 3.0 amb - {m} - {s['name']}:")
            run_with_trials(run_phase_30, s, kw, m, True)

    print("\n" + "=" * 60)
    print(f"ALL COMPLETE: {len(all_results['experiments'])} runs ({run_count}/{total_runs})")
    print("=" * 60)
    return all_results

print("✓ Runner defined (45 × 3 = 135 runs)")

In [ ]:
results = run_all()

## Save Results

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'paper4_v5_{timestamp}.json'
with open(filename, 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"✓ Saved: {filename} ({len(results['experiments'])} runs)")
try:
    from google.colab import files
    files.download(filename)
except: pass

## Analysis: Mean ± Std

In [ ]:
import numpy as np
from collections import defaultdict

# Group experiments by (phase, model, scenario, mode)
groups = defaultdict(list)
for e in results['experiments']:
    key = (e['phase'], e['model'], e['scenario'], e.get('mode', ''))
    groups[key].append(e)

def stats(values):
    if not values: return 0, 0
    return float(np.mean(values)), float(np.std(values))

models = ['claude', 'gpt', 'gemini']

# ============================================================
print("=" * 70)
print("[1] DESIGN SPACE (Bank) — mean ± std across 3 trials")
print("=" * 70)
print(f"  {'Phase':<8s} | {'Claude':>18s} | {'GPT':>18s} | {'Gemini':>18s}")
print("-" * 72)
for phase in ['1.0', '1.3', '1.5', '1.6', '2.0']:
    row = f"  {phase:<8s} |"
    for m in models:
        exps = groups.get((phase, m, 'bank', ''), [])
        toks = [e['total_tokens'] for e in exps]
        mu, sd = stats(toks)
        row += f" {mu:6.0f} ± {sd:5.1f}   |"
    print(row)
for mode_l, sname, mode in [('3.0 exp', 'bank', 'explicit'), ('3.0 amb', 'bank_ambiguous', 'ambiguous')]:
    row = f"  {mode_l:<8s} |"
    for m in models:
        exps = groups.get(('3.0', m, sname, mode), [])
        toks = [e['total_tokens'] for e in exps]
        mu, sd = stats(toks)
        row += f" {mu:6.0f} ± {sd:5.1f}   |"
    print(row)

# ============================================================
print("\n" + "=" * 70)
print("[2] TOKEN REDUCTION: 1.0 → 1.5 (Bank)")
print("=" * 70)
for m in models:
    e10 = groups.get(('1.0', m, 'bank', ''), [])
    e15 = groups.get(('1.5', m, 'bank', ''), [])
    mu10, sd10 = stats([e['total_tokens'] for e in e10])
    mu15, sd15 = stats([e['total_tokens'] for e in e15])
    if mu10 > 0:
        red = (mu10 - mu15) / mu10 * 100
        print(f"  {m:8s}: {mu10:.0f}±{sd10:.0f} → {mu15:.0f}±{sd15:.0f} ({red:.1f}% reduction)")

# ============================================================
print("\n" + "=" * 70)
print("[3] PHASE 1.5 STABILITY (avg tok/turn, mean ± std)")
print("=" * 70)
print(f"  {'Model':<10s} {'Bank(5t)':>16s} {'Spring(10t)':>16s} {'Court(12t)':>16s}")
for m in models:
    row = f"  {m:<10s}"
    for s in ['bank', 'spring', 'court']:
        exps = groups.get(('1.5', m, s, ''), [])
        avgs = [e['avg_per_turn'] for e in exps]
        mu, sd = stats(avgs)
        row += f" {mu:6.1f} ± {sd:4.1f}   "
    print(row)

# ============================================================
print("\n" + "=" * 70)
print("[4] SUCCESS / PARSE RATES (Bank, mean)")
print("=" * 70)
for phase, key_name in [('1.3', 'weight_parse_rate'), ('1.5', 'extraction_rate'), ('1.6', 'extraction_rate')]:
    rates = []
    for m in models:
        exps = groups.get((phase, m, 'bank', ''), [])
        vals = [e.get(key_name, 0) for e in exps]
        mu, _ = stats(vals)
        rates.append(f"{m}={mu*100:.0f}%")
    print(f"  Phase {phase}: {', '.join(rates)}")

# ============================================================
print("\n" + "=" * 70)
print("[5] PHASE 1.3 WEIGHT VOLATILITY (mean abs Δ per turn)")
print("=" * 70)
for m in models:
    exps = groups.get(('1.3', m, 'bank', ''), [])
    vols = [e.get('weight_volatility', 0) for e in exps]
    mu, sd = stats(vols)
    print(f"  {m:8s}: {mu:.3f} ± {sd:.3f}")
print("  (Phase 1.5 controlled step = 0.4 per turn by design)")

# ============================================================
print("\n" + "=" * 70)
print("[6] PHASE 2.0 SEMANTIC COLLAPSE (single_rate, mean)")
print("=" * 70)
for m in models:
    exps = groups.get(('2.0', m, 'bank', ''), [])
    rates = [e.get('single_rate', 0) for e in exps]
    mu, sd = stats(rates)
    print(f"  {m:8s}: {mu*100:.0f}% ± {sd*100:.0f}%")

# ============================================================
print("\n" + "=" * 70)
print("[7] PHASE 1.5 OPERATOR DISTRIBUTION (sigma vs delta)")
print("=" * 70)
for s in ['bank', 'spring', 'court']:
    print(f"  {s.upper()}:")
    for m in models:
        exps = groups.get(('1.5', m, s, ''), [])
        sigmas = [e.get('sigma_count', 0) for e in exps]
        deltas = [e.get('delta_count', 0) for e in exps]
        s_mu, s_sd = stats(sigmas)
        d_mu, d_sd = stats(deltas)
        total = s_mu + d_mu
        s_pct = s_mu/total*100 if total > 0 else 0
        print(f"    {m:8s}: σ={s_mu:.1f}±{s_sd:.1f} ({s_pct:.0f}%) δ={d_mu:.1f}±{d_sd:.1f} ({100-s_pct:.0f}%)")

# ============================================================
print("\n" + "=" * 70)
print("[8] PHASE 3.0: EXPLICIT vs AMBIGUOUS (mean ± std)")
print("=" * 70)
for base in ['bank', 'spring', 'court']:
    print(f"  {base.upper()}:")
    for m in models:
        ee = groups.get(('3.0', m, base, 'explicit'), [])
        ea = groups.get(('3.0', m, f'{base}_ambiguous', 'ambiguous'), [])
        e_toks = [e['avg_per_turn'] for e in ee]
        a_toks = [e['avg_per_turn'] for e in ea]
        e_mu, e_sd = stats(e_toks)
        a_mu, a_sd = stats(a_toks)
        e_auto = [e['auto_rate'] for e in ee]
        a_auto = [e['auto_rate'] for e in ea]
        ea_mu, _ = stats(e_auto)
        aa_mu, _ = stats(a_auto)
        print(f"    {m:8s}: exp={e_mu:.1f}±{e_sd:.1f}/t(auto:{ea_mu*100:.0f}%) | amb={a_mu:.1f}±{a_sd:.1f}/t(auto:{aa_mu*100:.0f}%)")

# ============================================================
print("\n" + "=" * 70)
print("[9] PHASE 1.5 EXTRACTION RATES (all, mean)")
print("=" * 70)
for s in ['bank', 'spring', 'court']:
    for m in models:
        exps = groups.get(('1.5', m, s, ''), [])
        rates = [e['extraction_rate'] for e in exps]
        mu, sd = stats(rates)
        n_turns = exps[0]['turns'][-1]['turn'] if exps and exps[0]['turns'] else '?'
        print(f"  {m:8s} {s:8s}: {mu*100:.0f}% ± {sd*100:.0f}% (n={n_turns})")

print("\n" + "=" * 70)
print("ANALYSIS COMPLETE")